# 阶段 5：为 Vitis HLS 生成测试向量，并做 Python 侧参考验证

思路：**让 Python 当裁判**。

1. 用阶段 4 训练/量化好的模型，在 Python 里跑定点推理，得到每个测试向量的期望类别；
2. 把「原始特征 + 期望类别」写进 `src/tv_mlp.h`；
3. HLS 的 `mlp_engine_tb.cpp` 拿这个头文件做 C 仿真。

于是 `csim_design` 通过 == 「HLS 实现和 Python 参考逐位一致」。
这是阶段 5 **唯一**能证明硬件实现正确的手段，必须过。

In [ ]:
import os
import sys
import hashlib
import numpy as np

# 自动定位工程根目录：从当前目录往上找，直到看见 lib/video_setup.py。
# 这样 ai_video 无论放在 /home/xilinx/jupyter_notebooks/ai_video，
# 还是塞在 OV5640_capture_only/ai_video 里，notebook 都不用改路径。
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
    ROOT = os.path.dirname(ROOT)
STAGE4 = os.path.join(ROOT, "stage04_mlp_train")
STAGE5 = os.path.join(ROOT, "stage05_mlp_hls")
sys.path.insert(0, ROOT)
sys.path.insert(0, STAGE4)

import mlp_common as mc

# 1) 把量化后的权重头文件同步到 HLS 的 src 目录
src_h = os.path.join(ROOT, "weights", "mlp_weights.h")
dst_h = os.path.join(STAGE5, "src", "mlp_weights.h")
if not os.path.exists(src_h):
    raise FileNotFoundError("缺少 " + src_h + "，请先完成阶段 4")
os.makedirs(os.path.dirname(dst_h), exist_ok=True)
with open(src_h, encoding="utf-8") as f:
    header_text = f.read()
with open(dst_h, "w", encoding="utf-8") as f:
    f.write(header_text)
qmodel = mc.from_c_header(header_text)

# 权重的 sha256 指纹，下一步会写进 tv_mlp.h。
# script.tcl 开跑前会拿它对一遍，确保"测试向量"和"权重"是同一代
# （踩过的坑：两者不同代时 csim 会报一堆 MISMATCH，看着特别像 C 代码有 bug）
w_sha = hashlib.sha256(header_text.encode("utf-8")).hexdigest()
print("已同步权重头文件到:", dst_h)
print("权重指纹 sha256 =", w_sha[:16], "...")
print("INPUT_SHIFT = %d, Q_SHIFT = 8" % qmodel["input_shift"])


## 生成测试向量 tv_mlp.h

除了从训练数据里抽的样本，还额外加入**边界样本**（全 0、全最大），
用来暴露饱和/移位写错的问题。

In [ ]:
from lib import features_io

csv_path, kind = features_io.pick_features_csv(ROOT)
print("测试向量取自:", os.path.basename(csv_path), "(%s)" % kind)
X, y = features_io.load_csv(csv_path)

# 每类抽 40 条 + 4 条边界样本。
# 边界样本不是"有意义的数据"，而是把饱和、移位、下标约定这些边界情况逼出来，
# 让 Python 参考实现和 HLS 在这几个点上必须给出同一个答案。
picked = []
for k in sorted(set(y.tolist())):
    picked.extend(np.where(y == k)[0][:40].tolist())
vectors = [[int(round(float(v))) for v in X[i]] for i in picked]
for edge in ([0, 0, 0], [255, 255, 255], [1, 1, 1], [200, 250, 60]):
    vectors.append(list(edge))

# 期望类别 = Python 定点参考实现
expected = [mc.predict_fixed_raw(qmodel, v[0], v[1], v[2]) for v in vectors]

lines = []
lines.append("// auto-generated by 05_mlp_verify.ipynb\n")
lines.append("// AI_VIDEO_WEIGHTS_SHA256: %s\n" % w_sha)
# 上面那行是权重指纹；script.tcl 会拿它对 src/mlp_weights.h 校验。
# 注释故意写成 ASCII：这个头文件要给 gcc 编译，非 ASCII 注释没必要冒险。
lines.append("// (weights fingerprint; checked by script.tcl against src/mlp_weights.h)\n")
lines.append("#ifndef AI_VIDEO_TV_MLP_H\n#define AI_VIDEO_TV_MLP_H\n\n")
lines.append("#define TV_N %d\n\n" % len(vectors))
lines.append("static const short TV_X[TV_N][3] = {\n")
lines.append(",\n".join("    {%d,%d,%d}" % tuple(v) for v in vectors))
lines.append("\n};\n\n")
lines.append("static const int TV_Y[TV_N] = {\n    ")
lines.append(",".join(str(int(e)) for e in expected))
lines.append("\n};\n\n#endif\n")

tv_path = os.path.join(STAGE5, "src", "tv_mlp.h")
with open(tv_path, "w", encoding="utf-8") as f:
    f.write("".join(lines))

print("已写出:", tv_path)
print("测试向量数:", len(vectors))
from collections import Counter
print("期望类别分布:", dict(Counter(expected)))
print("")
print("下一步：在虚拟机里 cd 到 stage05_mlp_hls/hls，跑 vivado_hls -f script.tcl")


## 下一步：跑 HLS

把 `stage05_mlp_hls/` 整个目录拷到装了 Vitis HLS 2024.2 的 PC 上，然后：

```tcl
cd <ai_video>/stage05_mlp_hls/hls
vivado_hls -f script.tcl
```

脚本里的 `csim_design` 会打印：

```
TV_N=124  fail=0
CSIM PASSED: HLS 与 Python 定点参考完全一致
```

只要出现任何 `MISMATCH`，就说明 HLS 代码和 Python 的移位/下标约定不一致，
**不要往下做阶段 6**，先把 `mlp_engine.cpp` 和 `mlp_common.predict_fixed_raw` 对齐。

In [ ]:
# 顺带把 Python 侧前 10 条的明细打出来，方便和 csim 的 MISMATCH 对照
print("idx   raw(luma,contrast,sharpness)   期望类别")
for t in range(min(10, len(vectors))):
    v = vectors[t]
    print("%3d   (%4d, %4d, %4d)            %d" % (t, v[0], v[1], v[2], expected[t]))